In [1]:
import pandas as pd
import json
import re
from pathlib import Path
from pdf2image import convert_from_path
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials
from google.genai import errors as genai_errors
import mimetypes
from datetime import datetime

# ==========================
# CONFIGURATION (COMPANY VM)
# ==========================

base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjUxNjUzMzEsImlhdCI6MTc2NTE2MzUzMiwiYXV0aF90aW1lIjoxNzY1MTYzNTMxLCJqdGkiOiJhOGRjM2I4My1lODg1LTRiYzMtOGI3Ny00OWZmOGJiODM5NjIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6ImNlM2U1NDE5LTlkNGItNGY2Ny05NDkxLTllMTc0OTZmYmZhNyIsImF0X2hhc2giOiJzTktGb2I5ZS1sWE9DZUFObk1Uc25BIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiJjZTNlNTQxOS05ZDRiLTRmNjctOTQ5MS05ZTE3NDk2ZmJmYTciLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.LmOdwZB5ZaAoGljWD5_VDKfAkiXu_ZnJ3MZB9O8WI7SELSIjsWO6imlckcvWyxY2WW7XnhitHwQ8TbgXINnjP-BcPxz3RmxD023bEu70WnBqoVCzKKHGMnbUUpsP3Dp6DDZPvVv96s1W5H5A2GNy7mRWvJazlNhSIe7NKLqRuzPPjuj_weNd5Sal175gzNcTAj4T2ZIuR3S3J9r_nosdDbEKOeLbA2HIfHbwhr8iC8LPVRxl1acLACeYEr0PTMYEtDvOMYrPl9-AzF9LCUE0yPlzNVHmdfQQHSKIfWV28IHm1wFp0Q4UImOBvngI2DYJdjWI1nfWMVAAH3fTf3LKyQ"
credentials = Credentials(access_token)


client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)

MODEL_NAME = "gemini-2.5-pro"

print("Industry-Grade Connector Distance Extractor Ready (Company Gateway).")


# ==========================
# ENHANCED INDUSTRY-FOCUSED PROMPT
# ==========================

def get_industrial_connection_prompt():
    return """
    You are an Industrial Engineering Drawing Analysis AI specializing in wiring harness 
    design, cable routing documentation, and connector-level distance extraction for 
    manufacturing and assembly purposes.

    INPUT CONTEXT:
    A technical drawing page that may contain:
    - Wiring harness routing diagrams
    - Cable assembly drawings
    - Interconnect diagrams
    - P&ID schematics
    - Electrical distribution layouts
    - Connection tables / wire lists / cable schedules
    - Dimensional annotations showing routing distances

    PRIMARY OBJECTIVE:
    Extract CONNECTOR-TO-CONNECTOR routing information with physical distances for:
    - Manufacturing planning (cable cutting lists)
    - Assembly instructions (harness build documentation)
    - Installation guides (field routing specifications)
    - Maintenance documentation (service and repair)

    EXTRACTION REQUIREMENTS:

    1. CONNECTOR-LEVEL IDENTIFICATION (NO PIN DETAILS)
       Focus ONLY on component/connector assemblies as complete units:
       
       - from_component: Source connector/component identifier
         Examples: "J1", "X06", "CONN_A", "MOTOR_1", "PLC_RACK", "TB1"
       
       - to_component: Destination connector/component identifier
         Examples: "J2", "X05", "CONN_B", "PUMP_2", "SENSOR_3", "TB2"

       CRITICAL RULES:
       - Ignore individual pin numbers and terminal assignments
       - Focus on the connector housing/assembly as a single entity
       - If multiple wires run between same two connectors, treat as ONE connection
       - Use the primary/reference designator visible on the drawing

    2. WIRE/CABLE IDENTIFICATION
       Extract harness-level properties from tables or labels:
       
       - wire_id: Cable assembly or wire harness identifier
         Examples: "HARNESS-001", "W1-MAIN", "Cable_A", "H123"
       
       - wire_gauge: Conductor cross-section or AWG size
         Examples: "0.5 mm²", "1.5 mm2", "18 AWG", "2.5mm²"
         Note: Preserve exact notation including spaces and superscripts
       
       - wire_color: Cable jacket color or color code
         Examples: "BLACK", "BLK/RED", "Blue", "GRY"
       
       - cable_type: Cable construction type (if specified)
         Examples: "Shielded", "Multi-core", "Coax", "Twisted pair"

    3. ROUTING PATH SEGMENTATION
       Use 2D spatial reasoning to trace physical cable paths:
       
       - Follow the drawn route from source to destination connector
       - Identify routing segments between intermediate points:
         * Junction boxes or splice points
         * Cable trays or conduit entry/exit points
         * Support brackets or cable tie locations
         * Bends or routing direction changes
       
       - For EACH segment, extract the dimension annotation:
         Format: ["distance_1", "distance_2", ..., "distance_N"]
         
       Examples:
         Direct route: ["1200 mm"]
         Multi-segment: ["7 ft", "4 ft", "2 ft"]
         With junction: ["150 mm", "300 mm", "200 mm"]
       
       - Use null for segments without dimension labels
       - Use [] if NO routing dimensions are shown

    4. INTERMEDIATE ROUTING POINTS (OPTIONAL)
       If junction points or routing waypoints are labeled:
       
       - junction_points: List of intermediate point identifiers
         Examples: ["JB1", "SPLICE_A", "TRAY_ENTRY", "J-BOX-2"]
       
       This helps validate segment count and routing topology

    5. DEDUPLICATION LOGIC
       CRITICAL: Each physical connection appears ONLY ONCE in output
       
       - If "X06 to X05" exists, do NOT also report "X05 to X06"
       - Use alphabetical/numerical ordering for consistency
       - For bidirectional connections, choose one canonical direction

    6. TABLE-BASED EXTRACTION
       If connection tables are present:
       
       - Prioritize table data over diagram interpretation
       - Table headers to look for:
         * From/To, Source/Destination, Start/End
         * Wire ID, Cable Number, Harness ID
         * Length, Distance, Route Length
         * Gauge, Size, AWG
         * Color, Color Code
       
       - Cross-reference table data with diagram routing
       - Use diagram dimensions to supplement table length data

    7. OUTPUT FORMAT (STRICT JSON, NO MARKDOWN)
       Return ONLY valid JSON in this EXACT structure:

       {
         "connections": [
           {
             "from_component": "X06",
             "to_component": "X05",
             "wire_id": "HARNESS-001",
             "wire_gauge": "1.5 mm²",
             "wire_color": "BLACK",
             "cable_type": "Shielded",
             "segment_lengths": ["7 ft", "4 ft", "2 ft"],
             "junction_points": ["JB1", "JB2"],
             "total_segments": 3,
             "routing_notes": null
           }
         ],
         "table_detected": true,
         "table_location": "Lower right quadrant",
         "page_notes": "Main harness routing diagram"
       }

    QUALITY ASSURANCE RULES:
    - connections: MUST be present (empty array if no connections)
    - from_component and to_component: MUST NOT be null for valid connections
    - segment_lengths: MUST be array (use [] if no dimensions available)
    - total_segments: MUST equal length of segment_lengths array
    - No duplicate connections (enforce single direction A→B, not both A→B and B→A)
    - Preserve all units exactly as shown in drawing
    - Use null for truly missing/unreadable data (except arrays which use [])

    MANUFACTURING FOCUS:
    - Prioritize information needed for cable cutting and assembly
    - Distance accuracy is critical for material planning
    - Connector identification must match assembly BOM references
    - Segment breakdown enables progressive assembly validation
    - Junction points help with harness support structure planning

    ERROR HANDLING:
    - If page has NO usable connection data: 
      {"connections": [], "table_detected": false, "page_notes": "No connection data found"}
    - If table exists but is unreadable:
      {"connections": [], "table_detected": true, "page_notes": "Table present but illegible"}
    - Never return explanatory text outside JSON structure
    - Never include markdown code fences in JSON response
    """


# ==========================
# MODEL CALL WITH ENHANCED ERROR HANDLING
# ==========================

def analyze_image_industrial(image_path: str):
    """
    Industrial-grade image analysis with comprehensive error handling.
    Returns structured connection data focused on manufacturing use cases.
    """
    print(f"   → Analyzing: {image_path}...")

    try:
        with open(image_path, "rb") as f:
            image_bytes = f.read()
    except Exception as e:
        print(f"      [ERROR] Cannot read image file: {e}")
        return {"connections": [], "table_detected": False, "page_notes": "File read error"}

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[get_industrial_connection_prompt(), image_part],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
            ),
        )

        raw_text = response.text or "{}"

        # Strip code fences if present
        if "```json" in raw_text:
            raw_text = raw_text.split("```json")[1].split("```")[0]
        elif "```" in raw_text:
            raw_text = raw_text.split("```")[1].split("```")[0]

        data = json.loads(raw_text)

    except genai_errors.ClientError as e:
        print("      [ClientError] Gateway/auth/model issue")
        print(f"      Message: {getattr(e, 'message', str(e))}")
        return {"connections": [], "table_detected": False, "page_notes": "API error"}

    except json.JSONDecodeError as e:
        print(f"      [JSONDecodeError] Invalid JSON response: {e}")
        print(f"      First 300 chars: {raw_text[:300] if raw_text else 'Empty'}")
        return {"connections": [], "table_detected": False, "page_notes": "JSON parse error"}

    except Exception as e:
        print(f"      [Error] Unexpected analysis failure: {type(e).__name__}")
        print(f"      Detail: {str(e)}")
        return {"connections": [], "table_detected": False, "page_notes": "Analysis error"}

    # Validate and normalize structure
    if not isinstance(data, dict):
        data = {"connections": [], "table_detected": False}
    
    if "connections" not in data or not isinstance(data.get("connections"), list):
        data["connections"] = []
    
    if "table_detected" not in data:
        data["table_detected"] = False

    return data


# ==========================
# DEDUPLICATION WITH ENHANCED LOGIC
# ==========================

def normalize_connection_key(from_comp, to_comp):
    """
    Create normalized key for deduplication.
    Uses alphabetical/numerical sorting to ensure X06-X05 and X05-X06 
    produce identical keys.
    """
    if from_comp is None or to_comp is None:
        return None
    
    # Clean and normalize component names
    comp1 = str(from_comp).strip().upper()
    comp2 = str(to_comp).strip().upper()
    
    if not comp1 or not comp2:
        return None
    
    # Sort to create canonical ordering
    comp_list = sorted([comp1, comp2])
    return tuple(comp_list)


def deduplicate_connections(connections):
    """
    Remove duplicate connections while preserving first occurrence.
    Also validates connection data quality.
    """
    seen = set()
    unique_connections = []
    duplicates_removed = 0
    
    for conn in connections:
        from_comp = conn.get("From_Component")
        to_comp = conn.get("To_Component")
        
        # Skip invalid connections
        if not from_comp or not to_comp:
            print(f"      ⚠ Skipping invalid connection: {from_comp} → {to_comp}")
            continue
        
        # Create normalized key
        key = normalize_connection_key(from_comp, to_comp)
        if key is None:
            continue
        
        # Check for duplicate
        if key in seen:
            duplicates_removed += 1
            print(f"      → Removing duplicate: {from_comp} ↔ {to_comp}")
            continue
        
        seen.add(key)
        unique_connections.append(conn)
    
    if duplicates_removed > 0:
        print(f"   ✓ Removed {duplicates_removed} duplicate connections")
    
    return unique_connections


# ==========================
# ENHANCED PIPELINE WITH INDUSTRIAL FEATURES
# ==========================

def process_file_industrial(file_path: str, output_dir: str = None):
    """
    Industrial-grade processing pipeline for engineering drawings.
    
    Features:
    - Robust PDF conversion with detailed diagnostics
    - Per-page connection extraction with table detection
    - Automatic deduplication of reverse connections
    - Multi-sheet Excel output with summary analytics
    - Audit trail with processing metadata
    
    Args:
        file_path: Path to PDF or image file
        output_dir: Optional output directory (uses file location if None)
    """
    print(f"\n{'='*70}")
    print(f"INDUSTRIAL CONNECTOR DISTANCE EXTRACTION")
    print(f"{'='*70}")
    print(f"File: {file_path}")
    print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}")
    
    # Validate input file
    file_path_obj = Path(file_path)
    if not file_path_obj.exists():
        print(f"\n❌ ERROR: File not found: {file_path}")
        return
    
    ext = file_path_obj.suffix.lower()
    temp_images = []
    processing_errors = []

    # 1) PDF Conversion with enhanced error handling
    if ext == ".pdf":
        print("\n📄 PDF Conversion Phase")
        print("─" * 70)
        
        try:
            file_size_mb = file_path_obj.stat().st_size / (1024 * 1024)
            print(f"   File size: {file_size_mb:.2f} MB")
            
            if file_size_mb > 50:
                print(f"   ⚠ Warning: Large file may take significant time to process")
            
            print(f"   Converting to images (300 DPI)...")
            pages = convert_from_path(
                file_path,
                dpi=300,
                fmt='png',
                thread_count=1
            )
            
            if not pages or len(pages) == 0:
                raise ValueError("PDF conversion returned 0 pages")
            
            print(f"   ✓ Successfully converted {len(pages)} pages")
            
            for i, page in enumerate(pages, start=1):
                img_path = f"temp_conn_page_{i:03d}.png"
                page.save(img_path, "PNG")
                temp_images.append(img_path)
                print(f"      Page {i:2d} → {img_path}")
            
        except Exception as e:
            error_msg = f"PDF conversion failed: {type(e).__name__} - {str(e)}"
            print(f"\n❌ {error_msg}")
            processing_errors.append(error_msg)
            
            print("\n🔧 Troubleshooting Steps:")
            print("   1. Verify poppler-utils is installed:")
            print("      - Linux: sudo apt-get install poppler-utils")
            print("      - Mac: brew install poppler")
            print("      - Windows: Download from https://github.com/oschwartz10612/poppler-windows")
            print("   2. Check PDF is not corrupted (try opening manually)")
            print("   3. Verify PDF is not password-protected")
            print("   4. Ensure pdf2image is installed: pip install pdf2image pillow")
            return
    else:
        print("\n🖼️ Image File Processing")
        print("─" * 70)
        temp_images = [str(file_path)]
        print(f"   ✓ Processing single image: {file_path_obj.name}")

    # 2) Connection Extraction Phase
    print(f"\n📊 Connection Extraction Phase")
    print("─" * 70)
    print(f"Total pages to analyze: {len(temp_images)}\n")
    
    all_connections = []
    table_info = []
    page_statistics = []

    for page_idx, img in enumerate(temp_images, start=1):
        print(f"{'─'*70}")
        print(f"Page {page_idx}/{len(temp_images)}: {Path(img).name}")
        print(f"{'─'*70}")
        
        page_data = analyze_image_industrial(img)
        
        # Extract metadata
        table_detected = page_data.get("table_detected", False)
        table_location = page_data.get("table_location", "N/A")
        page_notes = page_data.get("page_notes", "")
        conns = page_data.get("connections", [])
        
        # Record table detection
        if table_detected:
            table_info.append({
                "Page": page_idx,
                "Location": table_location,
                "Connections": len(conns)
            })
            print(f"   ✓ Connection table detected: {table_location}")
        else:
            print(f"   ⚠ No connection table found")
        
        if page_notes:
            print(f"   Note: {page_notes}")
        
        # Validate connection data
        if not isinstance(conns, list):
            error_msg = f"Page {page_idx}: Invalid connection data format"
            processing_errors.append(error_msg)
            print(f"   ❌ {error_msg}")
            continue
        
        print(f"   → Extracted {len(conns)} connections")
        
        # Process each connection
        valid_connections = 0
        for conn_idx, c in enumerate(conns, start=1):
            from_comp = c.get("from_component")
            to_comp = c.get("to_component")
            wire_id = c.get("wire_id")
            wire_gauge = c.get("wire_gauge")
            wire_color = c.get("wire_color")
            cable_type = c.get("cable_type")
            segments = c.get("segment_lengths", [])
            junctions = c.get("junction_points", [])
            routing_notes = c.get("routing_notes")
            
            # Validate essential fields
            if not from_comp or not to_comp:
                print(f"      ⚠ [{conn_idx}] Skipping: Missing component identifiers")
                continue
            
            # Normalize segments
            if not isinstance(segments, list):
                segments = []
            cleaned_segments = [
                s.strip() for s in segments 
                if isinstance(s, str) and s.strip()
            ]
            
            # Normalize junctions
            if not isinstance(junctions, list):
                junctions = []
            cleaned_junctions = [
                j.strip() for j in junctions 
                if isinstance(j, str) and j.strip()
            ]
            
            # Build connection record
            all_connections.append({
                "Page": page_idx,
                "From_Component": str(from_comp).strip(),
                "To_Component": str(to_comp).strip(),
                "Wire_ID": wire_id,
                "Wire_Gauge": wire_gauge,
                "Wire_Color": wire_color,
                "Cable_Type": cable_type,
                "Total_Segments": len(cleaned_segments),
                "Segment_Lengths": cleaned_segments,
                "Junction_Points": cleaned_junctions,
                "Routing_Notes": routing_notes
            })
            
            valid_connections += 1
            
            # Print connection summary
            print(f"      [{conn_idx}] {from_comp} → {to_comp}")
            if wire_id or wire_gauge or wire_color:
                details = ", ".join(filter(None, [wire_id, wire_gauge, wire_color]))
                print(f"            Wire: {details}")
            if cleaned_segments:
                print(f"            Segments: {cleaned_segments}")
            if cleaned_junctions:
                print(f"            Via: {', '.join(cleaned_junctions)}")
        
        # Page statistics
        page_statistics.append({
            "Page": page_idx,
            "Table_Detected": table_detected,
            "Connections_Found": valid_connections,
            "Has_Dimensions": sum(1 for c in all_connections if c["Total_Segments"] > 0)
        })

    # 3) Deduplication Phase
    print(f"\n{'─'*70}")
    print("Deduplication Phase")
    print(f"{'─'*70}")
    
    connections_before = len(all_connections)
    all_connections = deduplicate_connections(all_connections)
    connections_after = len(all_connections)
    
    print(f"   Connections before: {connections_before}")
    print(f"   Connections after:  {connections_after}")
    print(f"   Duplicates removed: {connections_before - connections_after}")

    # 4) Validation and Summary
    print(f"\n{'='*70}")
    print("EXTRACTION SUMMARY")
    print(f"{'='*70}")
    print(f"Total Pages Processed:       {len(temp_images)}")
    print(f"Tables Detected:             {len(table_info)}")
    for t in table_info:
        print(f"   - Page {t['Page']}: {t['Location']} ({t['Connections']} connections)")
    print(f"Total Unique Connections:    {len(all_connections)}")
    
    connections_with_dims = sum(1 for c in all_connections if c["Total_Segments"] > 0)
    print(f"Connections with Dimensions: {connections_with_dims} ({connections_with_dims/len(all_connections)*100:.1f}%)" if all_connections else "Connections with Dimensions: 0")
    
    if processing_errors:
        print(f"\nProcessing Errors: {len(processing_errors)}")
        for err in processing_errors:
            print(f"   ⚠ {err}")

    if not all_connections:
        print("\n⚠ No valid connections extracted. No Excel file created.")
        
        # Cleanup temp files
        if ext == ".pdf":
            for t in temp_images:
                Path(t).unlink(missing_ok=True)
        
        return

    # 5) Excel Output Generation
    print(f"\n{'─'*70}")
    print("Excel Output Generation")
    print(f"{'─'*70}")

    # Determine maximum columns needed
    max_segments = max((c["Total_Segments"] for c in all_connections), default=0)
    max_junctions = max((len(c["Junction_Points"]) for c in all_connections), default=0)
    
    print(f"   Max segments per connection:  {max_segments}")
    print(f"   Max junctions per connection: {max_junctions}")

    # Flatten connection data
    flat_rows = []
    for conn in all_connections:
        base = {
            "Page": conn["Page"],
            "From_Component": conn["From_Component"],
            "To_Component": conn["To_Component"],
            "Wire_ID": conn["Wire_ID"],
            "Wire_Gauge": conn["Wire_Gauge"],
            "Wire_Color": conn["Wire_Color"],
            "Cable_Type": conn["Cable_Type"],
            "Total_Segments": conn["Total_Segments"],
        }
        
        # Add segment columns
        segs = conn["Segment_Lengths"]
        for idx in range(max_segments):
            col_name = f"Segment_{idx + 1}"
            base[col_name] = segs[idx] if idx < len(segs) else None
        
        # Add junction columns
        juncs = conn["Junction_Points"]
        for idx in range(max_junctions):
            col_name = f"Junction_{idx + 1}"
            base[col_name] = juncs[idx] if idx < len(juncs) else None
        
        base["Routing_Notes"] = conn["Routing_Notes"]
        flat_rows.append(base)

    # Create DataFrames
    df_connections = pd.DataFrame(flat_rows)
    
    # Summary sheet data
    summary_data = {
        "Metric": [
            "Extraction Date/Time",
            "Source File",
            "Total Pages",
            "Tables Detected",
            "Total Unique Connections",
            "Connections with Dimensions",
            "Unique From Components",
            "Unique To Components",
            "Max Segments per Connection",
            "Max Junctions per Connection",
            "Processing Errors"
        ],
        "Value": [
            datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            file_path_obj.name,
            len(temp_images),
            len(table_info),
            len(all_connections),
            connections_with_dims,
            df_connections["From_Component"].nunique(),
            df_connections["To_Component"].nunique(),
            max_segments,
            max_junctions,
            len(processing_errors)
        ]
    }
    df_summary = pd.DataFrame(summary_data)
    
    # Page statistics sheet
    df_page_stats = pd.DataFrame(page_statistics)
    
    # Table detection sheet
    if table_info:
        df_tables = pd.DataFrame(table_info)
    else:
        df_tables = pd.DataFrame({"Message": ["No connection tables detected"]})

    # Output filename
    if output_dir:
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
    else:
        output_path = file_path_obj.parent
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_filename = output_path / f"{file_path_obj.stem}_CONNECTOR_DISTANCES_{timestamp}.xlsx"

    # Write Excel with multiple sheets
    print(f"\n   Writing Excel file: {output_filename.name}")
    
    with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
        # Main connections sheet
        df_connections.to_excel(
            writer, 
            sheet_name="Connector_Distances", 
            index=False
        )
        
        # Summary sheet
        df_summary.to_excel(
            writer, 
            sheet_name="Summary", 
            index=False
        )
        
        # Page statistics
        df_page_stats.to_excel(
            writer, 
            sheet_name="Page_Statistics", 
            index=False
        )
        
        # Table detection info
        df_tables.to_excel(
            writer, 
            sheet_name="Tables_Detected", 
            index=False
        )
        
        # Processing errors (if any)
        if processing_errors:
            df_errors = pd.DataFrame({"Errors": processing_errors})
            df_errors.to_excel(
                writer, 
                sheet_name="Processing_Errors", 
                index=False
            )

    print(f"\n{'='*70}")
    print("✅ SUCCESS!")
    print(f"{'='*70}")
    print(f"Output file: {output_filename}")
    print(f"\nExcel Sheets:")
    print(f"   1. Connector_Distances  → {len(flat_rows)} connections")
    print(f"   2. Summary              → Extraction metadata")
    print(f"   3. Page_Statistics      → Per-page analysis")
    print(f"   4. Tables_Detected      → Table location info")
    if processing_errors:
        print(f"   5. Processing_Errors    → {len(processing_errors)} errors logged")

    # 6) Cleanup temporary files
    if ext == ".pdf":
        print(f"\n🧹 Cleaning up {len(temp_images)} temporary image files...")
        for t in temp_images:
            Path(t).unlink(missing_ok=True)
        print("   ✓ Cleanup complete")

    print(f"\n{'='*70}")
    print("Processing Complete")
    print(f"{'='*70}\n")


# ==========================
# ENTRY POINT & DIAGNOSTICS
# ==========================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("INDUSTRY-GRADE CONNECTOR DISTANCE EXTRACTOR")
    print("Optimized for Manufacturing & Assembly Documentation")
    print("="*70)
    
    # System diagnostics
    print("\n🔧 System Diagnostics:")
    print(f"   Working Directory: {Path.cwd()}")
    
    # Check dependencies
    try:
        from pdf2image import convert_from_path
        print("   ✓ pdf2image: Installed")
    except ImportError:
        print("   ❌ pdf2image: NOT INSTALLED")
        print("      Install: pip install pdf2image")
    
    try:
        import pandas as pd
        print("   ✓ pandas: Installed")
    except ImportError:
        print("   ❌ pandas: NOT INSTALLED")
    
    try:
        from openpyxl import Workbook
        print("   ✓ openpyxl: Installed")
    except ImportError:
        print("   ❌ openpyxl: NOT INSTALLED")
        print("      Install: pip install openpyxl")
    
    # Check for poppler utilities
    import subprocess
    import platform
    
    print("\n   Checking for Poppler utilities...")
    try:
        if platform.system() == "Windows":
            result = subprocess.run(
                ["where", "pdftoppm"], 
                capture_output=True, 
                text=True,
                timeout=5
            )
        else:
            result = subprocess.run(
                ["which", "pdftoppm"], 
                capture_output=True, 
                text=True,
                timeout=5
            )
        
        if result.returncode == 0:
            print("   ✓ Poppler: Found in PATH")
        else:
            print("   ⚠ Poppler: Not found in PATH")
            print("      Required for PDF processing")
            print("      Installation:")
            if platform.system() == "Windows":
                print("      → https://github.com/oschwartz10612/poppler-windows")
            elif platform.system() == "Darwin":
                print("      → brew install poppler")
            else:
                print("      → sudo apt-get install poppler-utils")
    except Exception as e:
        print(f"   ⚠ Could not check Poppler: {e}")
    
    print("\n" + "="*70)
    print("Ready to Process Engineering Drawings")
    print("="*70)
    print("\nUsage:")
    print("  process_file_industrial('drawing.pdf')")
    print("  process_file_industrial('drawing.pdf', output_dir='results')")
    print("\nExample:")
    print("  process_file_industrial('Harness_Assembly_Rev_A.pdf')")
    print("="*70 + "\n")

Industry-Grade Connector Distance Extractor Ready (Company Gateway).

INDUSTRY-GRADE CONNECTOR DISTANCE EXTRACTOR
Optimized for Manufacturing & Assembly Documentation

🔧 System Diagnostics:
   Working Directory: /home/jovyan/Data Extraction
   ✓ pdf2image: Installed
   ✓ pandas: Installed
   ✓ openpyxl: Installed

   Checking for Poppler utilities...
   ⚠ Poppler: Not found in PATH
      Required for PDF processing
      Installation:
      → sudo apt-get install poppler-utils

Ready to Process Engineering Drawings

Usage:
  process_file_industrial('drawing.pdf')
  process_file_industrial('drawing.pdf', output_dir='results')

Example:
  process_file_industrial('Harness_Assembly_Rev_A.pdf')



In [3]:
process_file_industrial("Master.png")


INDUSTRIAL CONNECTOR DISTANCE EXTRACTION
File: Master.png
Time: 2025-12-08 03:46:04

🖼️ Image File Processing
──────────────────────────────────────────────────────────────────────
   ✓ Processing single image: Master.png

📊 Connection Extraction Phase
──────────────────────────────────────────────────────────────────────
Total pages to analyze: 1

──────────────────────────────────────────────────────────────────────
Page 1/1: Master.png
──────────────────────────────────────────────────────────────────────
   → Analyzing: Master.png...
   ✓ Connection table detected: Multiple tables distributed across the page, adjacent to their respective connectors.
   Note: Main wiring harness diagram with detailed routing segments and connection tables.
   → Extracted 18 connections
      [1] X01 → X05
            Segments: ['80', '100', '100', '150', '120', '25']
      [2] X01 → X06
            Segments: ['80', '100', '100', '150', '45']
      [3] X02 → X05
            Segments: ['135', '40', '